# HealthDataManager to Admin Lakehouse Migration

This notebook migrates HealthDataManager (HDM) content from the HealthDataManager lakehouse to the Admin lakehouse and updates configuration files.

## Workflow Overview

1. **Configuration & Validation** - Set lakehouse IDs and validate
2. **Migrate DMH Folders** - Copy DMHCheckpoint, DMHConfiguration, DMHSampleData
3. **Verify Migration** - Check if folders exist at target location
4. **Update Configuration** - Automatically backup and update configuration file with new paths

## Prerequisites

- Admin Lakehouse ID (GUID)
- HealthDataManager Lakehouse ID (GUID)
- Access to both lakehouses
- Configuration file exists at specified path

## Instructions

1. Fill in `ADMIN_LAKEHOUSE_ID` and `HEALTHDATAMANAGER_LAKEHOUSE_ID` in the configuration cell
2. Run all cells in sequence
3. Review verification results to confirm all folders were migrated
4. Check the completion summary for any missing folders

---

## Import Libraries

In [ ]:
import json
from typing import Any, Dict, Optional, List
from notebookutils import mssparkutils

print("✓ Libraries imported")

---

## Configuration Parameters

Update the lakehouse IDs below before running the migration.

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# Required: Lakehouse IDs (must be filled by user)
ADMIN_LAKEHOUSE_ID: str = ""  # TODO: Fill with Admin Lakehouse GUID
HEALTHDATAMANAGER_LAKEHOUSE_ID: str = ""  # TODO: Fill with HealthDataManager Lakehouse GUID

# Configuration file paths
DEPLOYMENT_CONFIG_FOLDER: str = "Files/system-configurations"
DEPLOYMENT_CONFIG_FILENAME: str = "deploymentParametersConfiguration.json"

# Migration settings
HEALTH_DATA_MANAGER_ROOT: str = "Files/HealthDataManager"
DMH_FOLDERS: List[str] = ["DMHCheckpoint", "DMHConfiguration", "DMHSampleData"]
ONELAKE_HOST = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")

print("✓ Configuration loaded")

---

## Define Functions

All functions are defined below for validation, migration, and configuration updates.

In [ ]:
def validate_configuration(admin_id: str, hdm_id: str) -> None:
    """
    Validate that required lakehouse IDs are configured.
    
    Args:
        admin_id: Admin Lakehouse ID
        hdm_id: HealthDataManager Lakehouse ID
        
    Raises:
        ValueError: If any required ID is not set
    """
    print("=" * 80)
    print("VALIDATING CONFIGURATION")
    print("=" * 80)

    if not admin_id or admin_id.strip() == "":
        error_msg = "ADMIN_LAKEHOUSE_ID is not set"
        print(f"\n✗ ERROR: {error_msg}")
        print("  ACTION: Edit Cell 3 and provide Admin Lakehouse GUID")
        raise ValueError(error_msg)

    if not hdm_id or hdm_id.strip() == "":
        error_msg = "HEALTHDATAMANAGER_LAKEHOUSE_ID is not set"
        print(f"\n✗ ERROR: {error_msg}")
        print("  ACTION: Edit Cell 3 and provide HealthDataManager Lakehouse GUID")
        raise ValueError(error_msg)

    print("\n✓ Configuration validation passed")
    print(f"  Admin Lakehouse ID: {admin_id}")
    print(f"  HealthDataManager Lakehouse ID: {hdm_id}")
    print("=" * 80)

print("✓ validate_configuration() defined")

In [ ]:
def migrate_dmh_folders(
    hdm_lakehouse_id: str,
    onelake_host: str,
    health_data_manager_root: str,
    dmh_folders: List[str]
) -> Dict[str, Any]:
    """Migrate DMH folders from HealthDataManager lakehouse to Admin lakehouse.

    Source layout (HDM lakehouse):
      abfss://<wsId>@<host>/<hdmId>/<folder>

    Target layout (Admin lakehouse):
      abfss://<wsId>@<host>/<adminId>/<health_data_manager_root>/<folder>

    Returns a dict including per-folder failure reasons (if any).
    """
    ctx = notebookutils.runtime.context
    workspace_id = ctx["currentWorkspaceId"]
    admin_lake_house_id = ADMIN_LAKEHOUSE_ID

    source_root = f"abfss://{workspace_id}@{onelake_host}/{hdm_lakehouse_id}/"
    dest_root = f"abfss://{workspace_id}@{onelake_host}/{admin_lake_house_id}/{health_data_manager_root}"

    print("Starting DMH folder migration...")
    print(f"  Workspace ID: {workspace_id}")
    print(f"  Source root: {source_root}")
    print(f"  Target root: {dest_root}")

    folder_count = 0
    skipped_count = 0
    failure_reasons: Dict[str, str] = {}

    for folder in dmh_folders:
        src_folder = f"{source_root}{folder}"
        dest_parent = dest_root

        # 1) Check that the source exists
        try:
            if not mssparkutils.fs.exists(src_folder):
                msg = "Source folder does not exist"
                skipped_count += 1
                failure_reasons[folder] = msg
                print(f"  - {folder}: {msg}")
                continue
        except Exception as e:
            msg = f"Could not check existence: {e}"
            skipped_count += 1
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")
            continue

        # 2) Check if source folder is empty
        try:
            entries = mssparkutils.fs.ls(src_folder)
            if not entries:
                msg = "Source folder is empty; nothing to copy"
                skipped_count += 1
                failure_reasons[folder] = msg
                print(f"  - {folder}: {msg}")
                continue
        except Exception as e:
            msg = f"Could not list contents: {e}"
            skipped_count += 1
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")
            continue

        # 3) Perform the copy for non-empty folders
        try:
            mssparkutils.fs.fastcp(src_folder, dest_parent, True)
            folder_count += 1
            print(f"  - {folder}: Copied")
        except Exception as e:
            msg = f"Copy failed: {e}"
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")

    print("DMH folder migration complete.")
    print(f"  Folders copied: {folder_count}")
    print(f"  Folders skipped: {skipped_count}")

    return {
        "workspace_id": workspace_id,
        "admin_lake_house_name": admin_lake_house_id,
        "folder_count": folder_count,
        "skipped_count": skipped_count,
        "failure_reasons": failure_reasons,
    }

print("✓ migrate_dmh_folders() defined")

In [ ]:
def verify_migrated_folders(
    workspace_id: str,
    admin_lake_house_name: str,
    onelake_host: str,
    health_data_manager_root: str,
    dmh_folders: List[str]
) -> Dict[str, Any]:
    """Verify that DMH folders exist in the Admin lakehouse target location."""

    dest_root = f"abfss://{workspace_id}@{onelake_host}/{admin_lake_house_name}/{health_data_manager_root}"

    print("Verifying migrated folders...")
    print(f"  Target root: {dest_root}")

    success_folders: List[str] = []
    missing_folders: List[str] = []

    for folder in dmh_folders:
        dest_folder = f"{dest_root}/{folder}"
        try:
            if mssparkutils.fs.exists(dest_folder):
                success_folders.append(folder)
                print(f"  - {folder}: OK")
            else:
                missing_folders.append(folder)
                print(f"  - {folder}: NOT FOUND")
        except Exception as e:
            missing_folders.append(folder)
            print(f"  - {folder}: verification error: {e}")

    print("Verification complete.")

    return {
        "success_folders": success_folders,
        "missing_folders": missing_folders,
        "verification_passed": len(missing_folders) == 0
    }

print("✓ verify_migrated_folders() defined")

In [ ]:
def recursive_replace(obj: Any, string_mapping: Dict[str, str]) -> Any:
    """
    Recursively traverse and replace strings in any JSON data structure.

    Args:
        obj: JSON-serializable object (dict, list, str, int, float, bool, None)
        string_mapping: Dictionary mapping old_string -> new_string

    Returns:
        Updated object with all string replacements applied
    """
    if isinstance(obj, dict):
        result = {}
        for key, value in obj.items():
            new_key = key
            for old_str, new_str in string_mapping.items():
                new_key = new_key.replace(old_str, new_str)
            result[new_key] = recursive_replace(value, string_mapping)
        return result

    elif isinstance(obj, list):
        return [recursive_replace(item, string_mapping) for item in obj]

    elif isinstance(obj, str):
        result = obj
        for old_str, new_str in string_mapping.items():
            result = result.replace(old_str, new_str)
        return result

    else:
        return obj

print("✓ recursive_replace() defined")

In [ ]:
def update_configuration_file(
    workspace_id: str,
    admin_lake_house_name: str,
    onelake_host: str,
    config_folder: str,
    config_filename: str,
    hdm_lakehouse_id: str,
    admin_lakehouse_id: str
) -> str:
    """Update configuration file by replacing HDM lakehouse ID with Admin lakehouse ID.
    
    Creates a backup of the original file as _obsolete before updating.
    """

    # Construct file paths
    config_path = (
        f"abfss://{workspace_id}@{onelake_host}/{admin_lakehouse_id}/"
        f"{config_folder}/{config_filename}"
    )
    
    if config_filename.lower().endswith(".json"):
        base_name = config_filename[:-5]
        backup_filename = f"{base_name}_obsolete.json"
    else:
        backup_filename = f"{config_filename}_obsolete"
    
    backup_path = (
        f"abfss://{workspace_id}@{onelake_host}/{admin_lakehouse_id}/"
        f"{config_folder}/{backup_filename}"
    )

    print("Updating configuration file...")
    print(f"  Original: {config_path}")
    print(f"  Backup:   {backup_path}")

    # Read original configuration
    content = mssparkutils.fs.head(config_path, 10_000_000)
    json_data = json.loads(content)
    
    # Create backup of original configuration
    print("  Creating backup...")
    mssparkutils.fs.put(backup_path, content, overwrite=True)
    print("  ✓ Backup created")

    # Apply replacements
    replaceable_mapper = {
        f"{hdm_lakehouse_id}/DMHCheckpoint": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHCheckpoint",
        f"{hdm_lakehouse_id}/DMHConfiguration": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHConfiguration",
        f"{hdm_lakehouse_id}/DMHSampleData": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHSampleData",
    }
    
    updated_data = recursive_replace(json_data, replaceable_mapper)

    # Update original file in place
    updated_content = json.dumps(updated_data, indent=2, ensure_ascii=False)
    mssparkutils.fs.put(config_path, updated_content, overwrite=True)

    print("  ✓ Configuration file updated in place")
    print(f"  ✓ Original backed up as: {backup_filename}")

    return config_path

print("✓ update_configuration_file() defined")

---

## Main Orchestration Function

The main function orchestrates the entire migration workflow.

In [ ]:
def start_health_data_manager_migration() -> None:
    """Run HDM → Admin migration, verification, and config update with concise output."""

    print("Starting HealthDataManager → Admin Lakehouse migration...")
    print(f"  OneLake host: {ONELAKE_HOST}")
    print(f"  Admin Lakehouse ID: {ADMIN_LAKEHOUSE_ID}")
    print(f"  HDM Lakehouse ID: {HEALTHDATAMANAGER_LAKEHOUSE_ID}")
    print("Configuration:")
    print(f"  DMH folders: {', '.join(DMH_FOLDERS)}")
    print(f"  HDM root (relative): {HEALTH_DATA_MANAGER_ROOT}")
    print(f"  Config folder: {DEPLOYMENT_CONFIG_FOLDER}")
    print(f"  Config file: {DEPLOYMENT_CONFIG_FILENAME}")

    # Step 1: Validate configuration
    validate_configuration(ADMIN_LAKEHOUSE_ID, HEALTHDATAMANAGER_LAKEHOUSE_ID)

    # Step 2: Migrate DMH folders
    migration_result = migrate_dmh_folders(
        hdm_lakehouse_id=HEALTHDATAMANAGER_LAKEHOUSE_ID,
        onelake_host=ONELAKE_HOST,
        health_data_manager_root=HEALTH_DATA_MANAGER_ROOT,
        dmh_folders=DMH_FOLDERS
    )

    # Step 3: Verify migrated folders
    verification_result = verify_migrated_folders(
        workspace_id=migration_result["workspace_id"],
        admin_lake_house_name=migration_result["admin_lake_house_name"],
        onelake_host=ONELAKE_HOST,
        health_data_manager_root=HEALTH_DATA_MANAGER_ROOT,
        dmh_folders=DMH_FOLDERS
    )

    # Step 4: Update configuration file (creates backup and updates in place)
    output_path = update_configuration_file(
        workspace_id=migration_result["workspace_id"],
        admin_lake_house_name=migration_result["admin_lake_house_name"],
        onelake_host=ONELAKE_HOST,
        config_folder=DEPLOYMENT_CONFIG_FOLDER,
        config_filename=DEPLOYMENT_CONFIG_FILENAME,
        hdm_lakehouse_id=HEALTHDATAMANAGER_LAKEHOUSE_ID,
        admin_lakehouse_id=ADMIN_LAKEHOUSE_ID
    )

    # Summary
    print("\nMigration summary:")
    print(f"  Folders copied: {migration_result['folder_count']}")
    print(f"  Folders skipped: {migration_result['skipped_count']}")
    print(f"  Verified OK: {len(verification_result['success_folders'])}")
    print(f"  Missing/failed: {len(verification_result['missing_folders'])}")

    if verification_result['success_folders']:
        print("  Successful folders:")
        for folder in verification_result['success_folders']:
            print(f"    - {folder}")

    if verification_result['missing_folders']:
        print("  Missing/failed folders:")
        for folder in verification_result['missing_folders']:
            reason = migration_result.get('failure_reasons', {}).get(folder)
            if reason:
                print(f"    - {folder}: {reason}")
            else:
                print(f"    - {folder}: not found at destination")

    print(f"  Config file updated: {output_path.split('/')[-1]}")

print("✓ main() defined")

---

## Execute Migration

Run the main function to execute the complete migration workflow.

**⚠️ Important:** Ensure lakehouse IDs are configured in Cell 3 before running this cell.

In [ ]:
# Re-define migration and config functions with ID-only OneLake paths, then run migration

import json
from typing import Any, Dict, List
from notebookutils import mssparkutils

# We will mimic your example exactly:
#   source: abfss://<wsId>@<host>/<hdmId>/
#   target: abfss://<wsId>@<host>/<adminId>/<health_data_manager_root>


def migrate_dmh_folders(
    hdm_lakehouse_id: str,
    onelake_host: str,
    health_data_manager_root: str,
    dmh_folders: List[str]
) -> Dict[str, Any]:
    """Migrate DMH folders using ID-only paths.

    If the source folder exists but is empty, the copy is skipped and
    the reason "Source folder is empty; nothing to copy" is recorded.
    """
    ctx = notebookutils.runtime.context
    workspace_id = ctx["currentWorkspaceId"]
    admin_lake_house_id = ADMIN_LAKEHOUSE_ID

    source_root = f"abfss://{workspace_id}@{onelake_host}/{hdm_lakehouse_id}/"
    dest_root = f"abfss://{workspace_id}@{onelake_host}/{admin_lake_house_id}/{health_data_manager_root}"

    print("Starting DMH folder migration...")
    print(f"  Workspace ID: {workspace_id}")
    print(f"  Source root: {source_root}")
    print(f"  Target root: {dest_root}")

    folder_count = 0
    skipped_count = 0
    failure_reasons: Dict[str, str] = {}

    for folder in dmh_folders:
        src_folder = f"{source_root}{folder}"
        dest_parent = dest_root

        # 1) Check that the source exists
        try:
            if not mssparkutils.fs.exists(src_folder):
                msg = "Source folder does not exist"
                skipped_count += 1
                failure_reasons[folder] = msg
                print(f"  - {folder}: {msg}")
                continue
        except Exception as e:
            msg = f"Could not check existence: {e}"
            skipped_count += 1
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")
            continue

        # 2) Check if source folder is empty
        try:
            entries = mssparkutils.fs.ls(src_folder)
            if not entries:
                msg = "Source folder is empty; nothing to copy"
                skipped_count += 1
                failure_reasons[folder] = msg
                print(f"  - {folder}: {msg}")
                continue
        except Exception as e:
            msg = f"Could not list contents: {e}"
            skipped_count += 1
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")
            continue

        # 3) Perform the copy for non-empty folders
        try:
            mssparkutils.fs.fastcp(src_folder, dest_parent, True)
            folder_count += 1
            print(f"  - {folder}: Copied")
        except Exception as e:
            msg = f"Copy failed: {e}"
            failure_reasons[folder] = msg
            print(f"  - {folder}: {msg}")

    print("DMH folder migration complete.")
    print(f"  Folders copied: {folder_count}")
    print(f"  Folders skipped: {skipped_count}")

    return {
        "workspace_id": workspace_id,
        "admin_lake_house_name": admin_lake_house_id,
        "folder_count": folder_count,
        "skipped_count": skipped_count,
        "failure_reasons": failure_reasons,
    }


def verify_migrated_folders(
    workspace_id: str,
    admin_lake_house_name: str,
    onelake_host: str,
    health_data_manager_root: str,
    dmh_folders: List[str],
) -> Dict[str, Any]:
    """Verify DMH folders under target: abfss://<wsId>@<host>/<adminId>/<health_data_manager_root>/<folder>."""

    dest_root = f"abfss://{workspace_id}@{onelake_host}/{admin_lake_house_name}/{health_data_manager_root}"

    print("Verifying migrated folders...")
    print(f"  Target root: {dest_root}")

    success_folders: List[str] = []
    missing_folders: List[str] = []

    for folder in dmh_folders:
        dest_folder = f"{dest_root}/{folder}"
        try:
            if mssparkutils.fs.exists(dest_folder):
                success_folders.append(folder)
                print(f"  - {folder}: OK")
            else:
                missing_folders.append(folder)
                print(f"  - {folder}: NOT FOUND")
        except Exception as e:
            missing_folders.append(folder)
            print(f"  - {folder}: verification error: {e}")

    print("Verification complete.")

    return {
        "success_folders": success_folders,
        "missing_folders": missing_folders,
        "verification_passed": len(missing_folders) == 0,
    }


def update_configuration_file(
    workspace_id: str,
    admin_lake_house_name: str,
    onelake_host: str,
    config_folder: str,
    config_filename: str,
    hdm_lakehouse_id: str,
    admin_lakehouse_id: str
) -> str:
    """Update configuration file by replacing HDM lakehouse ID with Admin lakehouse ID.
    
    Creates a backup of the original file as _obsolete before updating.
    """

    # Construct file paths
    config_path = (
        f"abfss://{workspace_id}@{onelake_host}/{admin_lakehouse_id}/"
        f"{config_folder}/{config_filename}"
    )
    
    if config_filename.lower().endswith(".json"):
        base_name = config_filename[:-5]
        backup_filename = f"{base_name}_obsolete.json"
    else:
        backup_filename = f"{config_filename}_obsolete"
    
    backup_path = (
        f"abfss://{workspace_id}@{onelake_host}/{admin_lakehouse_id}/"
        f"{config_folder}/{backup_filename}"
    )

    print("Updating configuration file...")
    print(f"  Original: {config_path}")
    print(f"  Backup:   {backup_path}")

    # Read original configuration
    content = mssparkutils.fs.head(config_path, 10_000_000)
    json_data = json.loads(content)
    
    # Create backup of original configuration
    print("  Creating backup...")
    mssparkutils.fs.put(backup_path, content, overwrite=True)
    print("  ✓ Backup created")

    # Apply replacements
    replaceable_mapper = {
        f"{hdm_lakehouse_id}/DMHCheckpoint": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHCheckpoint",
        f"{hdm_lakehouse_id}/DMHConfiguration": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHConfiguration",
        f"{hdm_lakehouse_id}/DMHSampleData": f"{admin_lakehouse_id}/Files/HealthDataManager/DMHSampleData",
    }
    
    updated_data = recursive_replace(json_data, replaceable_mapper)

    # Update original file in place
    updated_content = json.dumps(updated_data, indent=2, ensure_ascii=False)
    mssparkutils.fs.put(config_path, updated_content, overwrite=True)

    print("  ✓ Configuration file updated in place")
    print(f"  ✓ Original backed up as: {backup_filename}")

    return config_path


# Orchestration: re-use existing validation + new functions above
start_health_data_manager_migration()

---

## ⚠️ Post-Migration Manual Steps for Existing Users

After the migration completes successfully, **existing users** must update the config notebook to ensure compatibility with the new folder structure.

### Step 1: Update Config Notebook

**Notebook:** `msft_config_notebook`

**To navigate to the code location:** copy and search for this text: "** # Workspace Config **" text in the msft_config_notebook and find the relevant code section.

**Change Required:**

Update the solution_name GUID with Admin Lakehouse GUID:

```python
solution_name = '<ADMIN_LAKEHOUSE_ID>' # Fill with Admin Lakehouse GUID
```

### Step 2: Update Config Notebook

**Notebook:** `msft_config_notebook`

**To navigate to the code location:** copy and search for this text: "** # Workload config **" text in the msft_config_notebook and find the relevant code section.

**Change Required:**

Update the is_config_in_workload flag from True to False:

```python
# BEFORE (old flag)
is_config_in_workload = True

# AFTER (new flag)
is_config_in_workload = False
```

### Step 3: Update Config Notebook

**Notebook:** `msft_config_notebook`

**To navigate to the code location:** copy and search for this text: "**Resolves correct path if the config files are in a lakehouse**" text in the msft_config_notebook to find the relevant code section.

**Change Required:**

Update the solution_name path construction:

```python
# BEFORE (old path)
if not is_config_in_workload:
    solution_name = f"{solution_name}/Files"

# AFTER (new path)
if not is_config_in_workload:
    solution_name = f"{solution_name}/Files/HealthDataManager"
```